**Lab 3: Control Flow for Data Cleaning**

**Problem Statement:
A healthcare analytics team is analyzing patient cardiovascular data. Before statistical
analysis, the dataset must be checked and cleaned because clinical data collected from
different sources may contain invalid values, missing observations, and extreme
measurements.
For this practical, use the resting blood pressure (trestbps) variable. To simulate
realistic data-entry problems, deliberately introduce a few negative BP values, missing
values, and extreme BP readings greater than 300 mmHg.**


**Name**: Yashraj Patil

**Roll no**: 23102A0071

**Department and Division**: CMPN-A

**Lab mentor**: Prof. Prakash Parmar

In [32]:
# STEP 0: Install required package (run once)
if (!require("microbenchmark")) install.packages("microbenchmark")
library(microbenchmark)

In [33]:
# STEP 1: Load the dataset
list.files("/content")   # confirm filename

df <- read.csv("/content/heart_disease_uci.csv")
str(df)
summary(df$trestbps)


[1] "cleaned_heart_data.csv" "heart_disease_uci.csv"  "sample_data"

'data.frame':	920 obs. of  16 variables:
 $ id      : int  1 2 3 4 5 6 7 8 9 10 ...
 $ age     : int  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : chr  "Male" "Male" "Male" "Male" ...
 $ dataset : chr  "Cleveland" "Cleveland" "Cleveland" "Cleveland" ...
 $ cp      : chr  "typical angina" "asymptomatic" "asymptomatic" "non-anginal" ...
 $ trestbps: int  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : int  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : logi  TRUE FALSE FALSE FALSE FALSE FALSE ...
 $ restecg : chr  "lv hypertrophy" "lv hypertrophy" "lv hypertrophy" "normal" ...
 $ thalch  : int  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : logi  FALSE TRUE TRUE FALSE FALSE FALSE ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : chr  "downsloping" "flat" "flat" "downsloping" ...
 $ ca      : int  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : chr  "fixed defect" "normal" "reversable defect" "normal" ...
 $ num     : int  0 2 1 0 0 0 3 0 2

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
    0.0   120.0   130.0   132.1   140.0   200.0      59 

In [31]:
# STEP 2: Inject a few negative BP values
set.seed(1)
neg_idx <- sample(which(!is.na(df$trestbps)), 3)
df$trestbps[neg_idx] <- -df$trestbps[neg_idx]

cat("\nAfter injecting negative values:\n")
print(summary(df$trestbps))


After injecting negative values:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
    0.0   120.0   130.0   132.1   140.0   200.0      59 


In [30]:
# TASK 1: BP-cleaning function using if-else
clean_bp <- function(value) {
  if (is.na(value)) {
    return(NA)
  } else if (value < 0) {
    return(NA)
  } else if (value > 250) {
    return(250)
  } else {
    return(value)
  }
}

df$trestbps_cleaned <- sapply(df$trestbps, clean_bp)

In [29]:
# TASK 2: Error handling with tryCatch()
safe_mean_bp <- function(x) {
  tryCatch({
    result <- mean(x, na.rm = TRUE)
    if (is.nan(result)) stop("No valid BP values to calculate mean.")
    return(result)
  }, error = function(e) {
    message("Warning: ", e$message)
    return(NA)
  })
}

safe_ratio <- function(numerator, denominator) {
  tryCatch({
    if (is.na(numerator) || is.na(denominator)) {
      stop("Missing value in numerator or denominator.")
    }
    if (denominator == 0) {
      stop("Denominator is zero.")
    }
    return(numerator / denominator)
  }, error = function(e) {
    message("Warning: ", e$message)
    return(NA)
  })
}

mean_bp <- safe_mean_bp(df$trestbps_cleaned)
cat("\nSafe mean BP:", mean_bp, "\n")

df$chol_bp_ratio <- mapply(safe_ratio, df$chol, df$trestbps_cleaned)


Safe mean BP: 132.0117 


In [28]:
# TASK 3: Loop-based vs vectorized + timing
loop_time <- system.time({
  invalid_loop <- c()
  for (val in df$trestbps) {
    if (is.na(val)) next
    if (val < 0 || val > 250) {
      invalid_loop <- c(invalid_loop, val)
    }
  }
})

vec_time <- system.time({
  invalid_vec <- df$trestbps[(df$trestbps < 0 | df$trestbps > 250) & !is.na(df$trestbps)]
})

cat("\nLoop-based execution time:\n"); print(loop_time)
cat("Vectorized execution time:\n"); print(vec_time)
cat("Invalid values found (loop):", length(invalid_loop), "\n")
cat("Invalid values found (vectorized):", length(invalid_vec), "\n")

bm <- microbenchmark(
  loop = {
    invalid_loop <- c()
    for (val in df$trestbps) {
      if (is.na(val)) next
      if (val < 0 || val > 250) invalid_loop <- c(invalid_loop, val)
    }
  },
  vectorized = {
    df$trestbps[(df$trestbps < 0 | df$trestbps > 250) & !is.na(df$trestbps)]
  },
  times = 50
)
cat("\nmicrobenchmark results:\n")
print(bm)


Loop-based execution time:
   user  system elapsed 
  0.005   0.000   0.005 
Vectorized execution time:
   user  system elapsed 
      0       0       0 
Invalid values found (loop): 3 
Invalid values found (vectorized): 3 

microbenchmark results:
Unit: microseconds
       expr      min       lq       mean    median       uq      max neval
       loop 4142.632 4389.381 5149.33634 4524.8910 5453.978 9282.700    50
 vectorized   18.608   20.622   31.44002   31.1445   36.548   75.478    50


In [27]:
# TASK 4: Validate the cleaned data
missing_count <- sum(is.na(df$trestbps_cleaned))
bp_min <- min(df$trestbps_cleaned, na.rm = TRUE)
bp_max <- max(df$trestbps_cleaned, na.rm = TRUE)
bp_mean <- mean(df$trestbps_cleaned, na.rm = TRUE)
bp_median <- median(df$trestbps_cleaned, na.rm = TRUE)

cat("\n--- Validation Summary ---\n")
cat("Missing BP values:", missing_count, "\n")
cat("Min BP:", bp_min, "\n")
cat("Max BP:", bp_max, "\n")
cat("Mean BP:", round(bp_mean, 2), "\n")
cat("Median BP:", bp_median, "\n")
cat("No negative values remain:", all(na.omit(df$trestbps_cleaned) >= 0), "\n")
cat("No values over 250 remain:", all(na.omit(df$trestbps_cleaned) <= 250), "\n")


--- Validation Summary ---
Missing BP values: 62 
Min BP: 0 
Max BP: 200 
Mean BP: 132.01 
Median BP: 130 
No negative values remain: TRUE 
No values over 250 remain: TRUE 


In [34]:
# Save cleaned dataset
write.csv(df, "/content/cleaned_heart_data.csv", row.names = FALSE)
cat("\nSaved cleaned_heart_data.csv to /content\n")


Saved cleaned_heart_data.csv to /content


In [25]:
# Conclusion
if (vec_time["elapsed"] <= loop_time["elapsed"]) {
  cat("\nConclusion: The vectorized approach was faster than the loop-based approach\n",
      "because R's vectorized operations are implemented in optimized C code and avoid\n",
      "the overhead of interpreting each iteration of an R-level for loop.\n")
} else {
  cat("\nConclusion: On this run the loop was comparable/faster, likely due to the\n",
      "small dataset size; on larger datasets vectorization is expected to consistently\n",
      "outperform loops.\n")
}


Conclusion: The vectorized approach was faster than the loop-based approach
 because R's vectorized operations are implemented in optimized C code and avoid
 the overhead of interpreting each iteration of an R-level for loop.
